# 1. Imports + TA-Lib (neu!)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import tpqoa
from datetime import datetime, timezone, timedelta
import time
import pickle
import warnings
import ta  # pip install ta
warnings.filterwarnings('ignore')


# 2. Daten laden (S5 bleibt, aber wir resamplen später)

In [ ]:
api = tpqoa.tpqoa(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\oanda.cfg')
data = api.get_history(instrument='EUR_USD', start='2025-06-01', end='2025-12-17', granularity='S5', price='M')
data.drop(['o','h','l','volume','complete'], axis=1, inplace=True)
data.rename(columns={'c': 'price'}, inplace=True)
data.dropna(inplace=True)
print(f"✅ Daten geladen: {len(data):,} Zeilen")
data.to_csv(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_five_secondsII.csv')
print(f"Daten geladen und gespeichert als 20251217_five_secondsII.csv: {len(data)} Zeilen")
data.tail()


# 3. NEUES Label + Features Engineering (KERNVERBESSERUNG!)

In [6]:
# 1. NEUES Label: Kumulierte Return über nächste 60 Sekunden (> Spread = +1)
spread_threshold = 0.00015  # 1.5 Pips EURUSD
data['future_return_60s'] = data['price'].pct_change(12).shift(-12)  # 60s = 12 * 5s
data['direction'] = np.where(data['future_return_60s'] > spread_threshold, 1,
                    np.where(data['future_return_60s'] < -spread_threshold, -1, 0))
data.dropna(inplace=True)

# 2. Returns berechnen
data['returns'] = np.log(data['price'] / data['price'].shift(1))

# 3. GUTE FEATURES (statt roher Lags!)
data['sma_short'] = data['price'].rolling(10).mean() / data['price'] - 1      # 50s Trend
data['sma_long'] = data['price'].rolling(50).mean() / data['price'] - 1       # 4min Trend
data['vol_short'] = data['returns'].rolling(20).std()                         # Vol 100s
data['vol_ratio'] = data['returns'].rolling(10).std() / data['returns'].rolling(50).std()
data['rsi'] = ta.momentum.RSIIndicator(data['price'], window=20).rsi()         # RSI(20)
data['zscore'] = (data['price'] - data['price'].rolling(20).mean()) / data['price'].rolling(20).std()
data['hour'] = data.index.hour
data['is_eu_session'] = ((data['hour'] >= 7) & (data['hour'] <= 16)).astype(int)
data['is_us_session'] = ((data['hour'] >= 13) & (data['hour'] <= 22)).astype(int)

# Nur sinnvolle Features behalten
feature_cols = ['sma_short', 'sma_long', 'vol_short', 'vol_ratio', 'rsi', 'zscore',
                'hour', 'is_eu_session', 'is_us_session']
data = data[['price', 'returns', 'direction'] + feature_cols].dropna()

print("Neue Features:", feature_cols)
print("Klassenverteilung:", data['direction'].value_counts())
data.tail()


Neue Features: ['sma_short', 'sma_long', 'vol_short', 'vol_ratio', 'rsi', 'zscore', 'hour', 'is_eu_session', 'is_us_session']
Klassenverteilung: direction
 0    1793267
-1     149545
 1     146519
Name: count, dtype: int64


,price,returns,direction,sma_short,sma_long,vol_short,vol_ratio,rsi,zscore,hour,is_eu_session,is_us_session
time,,,,,,,,,,,,
2025-12-16 23:56:00,1.17494,0.000000,0,0.000023,8.851516e-06,0.000008,0.777230,40.455413,-1.527523,23,0,0
2025-12-16 23:56:05,1.17495,0.000009,0,0.000012,3.404400e-07,0.000008,0.939138,44.885927,-0.998369,23,0,0
2025-12-16 23:56:20,1.17494,-0.000009,0,0.000018,8.681294e-06,0.000008,0.930070,41.625687,-1.622349,23,0,0
2025-12-16 23:56:30,1.17494,0.000000,0,0.000014,8.681294e-06,0.000008,0.834794,41.625687,-1.622349,23,0,0
2025-12-16 23:56:35,1.17495,0.000009,0,0.000003,3.404400e-07,0.000008,0.930074,45.973760,-0.946543,23,0,0


# 4. Train/Test Split + Training (mit RandomForest!)

In [7]:
from sklearn.model_selection import train_test_split

# Train/Test Split (chronologisch!)
split_idx = int(0.8 * len(data))
train = data.iloc[:split_idx]
test = data.iloc[split_idx:]

X_train = train[feature_cols]
y_train = train['direction']
X_test = test[feature_cols]
y_test = test['direction']

# RandomForest statt LogReg (besser für Interaktionen!)
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# Prediction + Hit Ratio
train['pred'] = rf.predict(X_train)
test['pred'] = rf.predict(X_test)

hit_ratio_train = (train['pred'] == train['direction']).mean()
hit_ratio_test = (test['pred'] == test['direction']).mean()
print(f"Hit Ratio Train: {hit_ratio_train:.2%}")
print(f"Hit Ratio Test:  {hit_ratio_test:.2%}")


Hit Ratio Train: 59.87%
Hit Ratio Test:  70.86%


In [12]:
class MLTrader(tpqoa.tpqoa):
    def __init__(self, config_file, instrument, barlength, model, units=10000):
        super().__init__(config_file)
        self.instrument = instrument
        self.barlength = pd.to_timedelta(barlength)
        self.tick_data = pd.DataFrame()
        self.rawdata = pd.DataFrame()
        self.lastbar = None
        self.units = units
        self.position = 0
        self.profits = []
        self.ticks = 0
        self.model = model
        self.feature_cols = ['sma_short', 'sma_long', 'vol_short', 'vol_ratio', 'rsi', 'zscore',
                           'hour', 'is_eu_session', 'is_us_session']
        print(f"MLTrader initialized: {instrument}, {barlength}")

    def get_most_recent(self, days=5):
        while True:
            time.sleep(2)
            now = datetime.now(timezone.utc).replace(tzinfo=None)
            now = now - timedelta(microseconds=now.microsecond)
            past = now - timedelta(days=days)
            df = self.get_history(instrument=self.instrument, start=past, end=now,
                                granularity='S5', price='M', localize=False)['c'].dropna().to_frame()
            df.rename(columns={'c': self.instrument}, inplace=True)
            self.rawdata = df.tail(100).copy()  # Nur letzte 100 für Features
            self.lastbar = self.rawdata.index[-1]
            if pd.Timestamp.now(tz='UTC') - self.lastbar < self.barlength:
                break
        print(f"Initial data: {len(self.rawdata)} bars, last: {self.lastbar}")

    def on_success(self, time, bid, ask):
        self.ticks += 1
        if self.ticks % 50 == 0:  # Weniger Spam
            print(self.ticks, end=" ")
        recent_tick = pd.to_datetime(time)
        df = pd.DataFrame({self.instrument: (ask+bid)/2}, index=[recent_tick])
        self.tick_data = pd.concat([self.tick_data, df])

        if recent_tick - self.lastbar >= self.barlength and len(self.tick_data) > 50:
            self.resample_and_join()
            self.define_strategy()
            self.execute_trades()

    def resample_and_join(self):
        # Resample Ticks zu Bars
        resampled = self.tick_data.resample(self.barlength, label='right').last().dropna()
        if len(resampled) == 0:
            return  # Keine neuen Bars

        # An rawdata anhängen
        self.rawdata = pd.concat([self.rawdata, resampled])
        self.rawdata = self.rawdata.tail(200)  # Nur letzte 200 behalten (Memory)

        # Features berechnen (mit Sicherheitsprüfungen)
        if len(self.rawdata) > 50:  # Mindestens 50 Bars für Rolling
            self.rawdata['returns'] = np.log(self.rawdata[self.instrument] / self.rawdata[self.instrument].shift(1))

            # Safe Rolling (fillna=0 für NaNs)
            self.rawdata['sma_short'] = self.rawdata[self.instrument].rolling(10, min_periods=5).mean().fillna(method='bfill') / self.rawdata[self.instrument] - 1
            self.rawdata['sma_long'] = self.rawdata[self.instrument].rolling(50, min_periods=20).mean().fillna(method='bfill') / self.rawdata[self.instrument] - 1
            self.rawdata['vol_short'] = self.rawdata['returns'].rolling(20, min_periods=10).std().fillna(0)
            self.rawdata['vol_ratio'] = (self.rawdata['returns'].rolling(10, min_periods=5).std() /
                                       self.rawdata['returns'].rolling(50, min_periods=20).std()).fillna(1)

            # RSI mit ta (sicherer)
            try:
                self.rawdata['rsi'] = ta.momentum.RSIIndicator(self.rawdata[self.instrument], window=20).rsi()
            except:
                self.rawdata['rsi'] = 50  # Neutral bei Fehler

            self.rawdata['zscore'] = (self.rawdata[self.instrument] -
                                    self.rawdata[self.instrument].rolling(20, min_periods=10).mean()) / \
                                   self.rawdata[self.instrument].rolling(20, min_periods=10).std()
            self.rawdata['hour'] = self.rawdata.index.hour
            self.rawdata['is_eu_session'] = ((self.rawdata['hour'] >= 7) & (self.rawdata['hour'] <= 16)).astype(int)
            self.rawdata['is_us_session'] = ((self.rawdata['hour'] >= 13) & (self.rawdata['hour'] <= 22)).astype(int)

            self.lastbar = self.rawdata.index[-1]
            print(f"\n✅ New bar: {self.lastbar} | Shape: {self.rawdata.shape}")

    def define_strategy(self):
        if len(self.rawdata) < 10:
            return

        df = self.rawdata.copy()
        X = df[self.feature_cols].fillna(0)
        df['position'] = self.model.predict(X)
        self.rawdata.loc[df.index, 'position'] = df['position'].values[-len(self.rawdata):]
        print(f"Position: {self.rawdata['position'].iloc[-1]}")

    def execute_trades(self):
        current_signal = self.rawdata['position'].iloc[-1]
        if self.position == 0 and current_signal == 1:
            self.close_all()
            self.buy(size=self.units // 1000)  # Kleinere Units für Test
            self.position = 1
            print("LONG")
        elif self.position == 1 and current_signal == -1:
            self.close_all()
            self.sell(size=self.units // 1000)
            self.position = -1
            print("SHORT")
        elif self.position == -1 and current_signal == 1:
            self.close_all()
            self.position = 0
            print("FLAT")


In [13]:
# MLTrader mit neuen Features + RF-Modell
trader = MLTrader(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\oanda.cfg',
                  "EUR_USD", "5min", rf, units=10000)

trader.get_most_recent()
trader.stream_data(trader.instrument, stop=900)


MLTrader initialized: EUR_USD, 5min
Initial data: 100 bars, last: 2025-12-17 13:35:25+00:00
50 100 150 200 250 300 350 400 450 500 550 600 650 
✅ New bar: 2025-12-17 13:45:00+00:00 | Shape: (102, 11)


ValueError: shape mismatch: value array of shape (102,) could not be broadcast to indexing result of shape (104,)